# Preference Optimization: DPO vs ORPO vs KTO vs IPO

## The Landscape (2024)

Starting from DPO (2023), a wave of variants emerged:

| Algorithm | Year | Key Innovation | Reference Model |
|-----------|------|---------------|----------------|
| DPO       | 2023 | Direct optimization from RL closed form | Required |
| IPO       | 2024 | Adds regularization, prevents overfitting | Required |
| ORPO      | 2024 | No reference model, combines SFT+pref | Not needed |
| KTO       | 2024 | Binary labels only, Prospect Theory | Required |
| SimPO     | 2024 | Length-normalized reward, no ref model | Not needed |

All are available in TRL via `DPOConfig(loss_type=...)` except KTO which uses `KTOTrainer`.

In [ ]:
!pip install -U trl transformers datasets peft -q
import trl; print(f"TRL {trl.__version__}")

# Show all available loss types in this TRL version
from trl import DPOConfig
import inspect
sig = inspect.signature(DPOConfig.__init__)
print("\nDPOConfig available loss_types (check trl docs for your version):")
# Trigger an invalid loss type to see the valid ones
try:
    c = DPOConfig(output_dir="/tmp", loss_type="_invalid_")
    c  # won't get here
except ValueError as e:
    print(str(e))

In [ ]:
import torch, gc
from trl import DPOConfig, DPOTrainer
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset
from peft import LoraConfig

gc.collect(); torch.cuda.empty_cache()

model_id = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Shared preference dataset
pref_data = [
    {"prompt": "What is LoRA?",
     "chosen": "LoRA adds trainable rank-decomposition matrices to frozen weights, cutting trainable parameters by 99%.",
     "rejected": "LoRA is a popular Japanese animation series."},
    {"prompt": "Explain attention.",
     "chosen": "Attention(Q,K,V) = softmax(QK^T/√d_k)·V — tokens attend to relevant context regardless of distance.",
     "rejected": "Attention means focusing carefully on what someone says."},
    {"prompt": "What is quantization?",
     "chosen": "Quantization reduces weight precision (e.g. FP16→INT4), shrinking model size 4× with <2% quality loss using methods like GPTQ/AWQ.",
     "rejected": "Quantization is a music theory concept about dividing beats into equal parts."},
] * 8

dataset = Dataset.from_list(pref_data)
print(f"Dataset: {len(dataset)} preference pairs")

# Test all available loss types
loss_types = ["sigmoid", "ipo", "apo_zero"]  # stable ones across TRL versions
results = {}

for loss_type in loss_types:
    gc.collect(); torch.cuda.empty_cache()
    print(f"\nTraining with loss_type='{loss_type}'...")
    
    model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map="auto")
    lora = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj","v_proj"], task_type="CAUSAL_LM")
    
    config = DPOConfig(
        output_dir=f"/tmp/dpo_{loss_type}",
        loss_type=loss_type,
        beta=0.1,
        num_train_epochs=1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=5e-6,
        max_length=192,
        fp16=True,
        logging_steps=100,
        save_strategy="no",
        report_to="none",
    )
    
    trainer = DPOTrainer(model=model, ref_model=None, args=config, train_dataset=dataset, peft_config=lora)
    r = trainer.train()
    results[loss_type] = r.training_loss
    print(f"  {loss_type}: final loss = {r.training_loss:.4f}")
    del model, trainer

print("\n=== Results ===")
for lt, loss in results.items():
    desc = {"sigmoid": "Standard DPO", "ipo": "IPO (regularized)", "apo_zero": "APO-Zero"}
    print(f"  {lt:15s} ({desc.get(lt,''):<20s}): loss={loss:.4f}")
print("\n✅ Comparison complete")

## Decision Guide: Which Algorithm to Use?

```
Do you have paired data (chosen + rejected)?
├── YES → Do you have GPU memory for 2× model?
│   ├── YES → Use DPO (most stable, best understood)
│   │         └── Or IPO if DPO shows reward hacking
│   └── NO  → Use ORPO (no reference model, same dataset format)
│
└── NO → Do you have binary labels (thumbs up/down)?
    ├── YES → Use KTO
    └── NO  → Collect preference data first (use annotation tools or GPT-4)
```

**Memory guide (for 7B model on A100 40GB):**
- DPO: ~28GB (policy 14GB + reference 14GB)
- ORPO: ~14GB (policy only, no reference)
- KTO: ~28GB (policy + reference)

**Quality guide:** All methods achieve similar quality at 1-30B scale. Differences emerge at 70B+.